# Ranking predictions — models and saved results

**Question (a)**: the day before a season starts, predict the final regular-season ranking of every
team *inside its conference* (two rankings of ~15 teams per season, ordered by points).

This notebook is the **only place where models are trained**. It uses a deliberately small set:

| Family | Model | Why it is kept |
|---|---|---|
| Naive | Random ranking | floor with no skill (scale of the metrics) |
| Naive | Previous available ranking | strongest simple history baseline |
| External | Ridge + external | regularised linear model on the selected features |
| External | Random forest + external | non-linear challenger on the same features |
| Blend | Naive + best model | weighted average of last season's ranking and the best trained model |

So: **2 naive baselines + 2 models + 1 blend**. The variants tested during the project
(gradient boosting, ensemble, pairwise ranking) are kept in the git history; older notebooks are in
`archive/` for traceability.

The random forest is deliberately **shallow**: the training window is small (9 seasons × ~28 teams),
and a deeper forest fits 1995–2003 better but overfits the later period. A small dev-only grid
confirmed this (the best dev configuration reached 2.59 on 1995–2003 but 3.65 on 2005–2009), so we
keep a conservative depth. The clean test is never used to choose the hyper-parameters.

The blend is the honest way to try to beat the naive baseline: its weight is chosen on the
selection seasons (1995–2004) only, then frozen for both splits. It is not tuned on the test.

Everything is evaluated with the same walk-forward protocol:

> for each test season *Y*, train on every season `< Y` and predict *Y*.

The target is the **points percentage** of the season (``Pts / (2 * GP)``), expressed as a
league-wide z-score. Points percentage is comparable across 80-game, 82-game and shortened
(48-game) seasons, unlike raw points or wins.

**Leak policy.** No feature uses information from the season being predicted:

* every team-performance feature is lagged (season *Y-1* predicts *Y*);
* coach columns (`coach_changed`, tenure, interim, number of coaches) are excluded — they are the
  target of the separate coach-change task;
* current-season roster composition is excluded — the roster is only known after the season
  (in-season arrivals).

Two splits are stored:

* `clean` — 2005–2009, never used for design decisions;
* `backtest` — 1995–2009 (except the 2004 lockout), a stability/power check. Seasons 1995–2004
  were used for feature selection, so this split is **not** a clean out-of-sample test.

At the end, the notebook writes:

```
results/predictions.csv        one row per (split, model, season, team)
results/metrics_by_season.csv  one row per (split, model, season, conference)
results/summary.csv            one row per (split, model)
results/run_info.json          features, years, model descriptions
```

`ranking_results_analysis.ipynb` only reads these files; it never trains a model.

## 1. Data and features

* Base dataset: `data_prepared/teams.csv`, prepared by `data_preparation.ipynb`.
* External features: built by `ranking_features.py` from the raw tables.
* Selected features: `selected_features.json`, produced by `ranking_feature_analysis.ipynb`.

The external models use the selected leak-free features (history + shots/luck + any other block
kept by the selection). Both models share the same feature set; only the inductive bias changes.

In [1]:
import json
import sys
import platform
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import kendalltau
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor

# Make the helper modules importable whether the notebook is run from the project root
# or from the ranking/ directory.
for candidate in [Path.cwd(), Path.cwd() / "ranking"]:
    if (candidate / "ranking_utils.py").exists():
        sys.path.insert(0, str(candidate))
        break

import ranking_utils as ru
import ranking_features as rf

RS = ru.RANDOM_STATE
CLEAN_YEARS = [2005, 2006, 2007, 2008, 2009]
BACKTEST_YEARS = [y for y in range(1995, 2010) if y != 2004]
SPLITS = {"clean": CLEAN_YEARS, "backtest": BACKTEST_YEARS}

# ---- datasets --------------------------------------------------------------
teams = ru.load_teams()
df = ru.build_dataset(teams)

selection = json.loads(ru.SELECTED_FEATURES_PATH.read_text())
BASE_FEATURES = selection["base_features"]
EXT_FEATURES = selection["features"]
SELECTED_GROUPS = selection["selected_groups"]

ext, EXT_GROUPS = rf.build_external_features(ru.DATA_DIR)
df_ext = df.merge(ext.drop(columns=["franchID"]), on=["year", "tmID"], how="left")
EXT_COLS = [c for cols in EXT_GROUPS.values() for c in cols]
df_ext[EXT_COLS] = df_ext[EXT_COLS].fillna(0.0)

assert len(df_ext) == len(df), "External merge duplicated team-seasons"
assert not df[BASE_FEATURES].isna().any().any(), "NaN in base features"
assert not df_ext[EXT_FEATURES].isna().any().any(), "NaN in external features"

print(f"Team-seasons: {len(df)} | clean years: {CLEAN_YEARS}")
print(f"Selected groups: {SELECTED_GROUPS}")
print(f"Base features ({len(BASE_FEATURES)}): {BASE_FEATURES}")
print(f"External features ({len(EXT_FEATURES)}): {EXT_FEATURES}")

Team-seasons: 732 | clean years: [2005, 2006, 2007, 2008, 2009]
Selected groups: ['prev_points', 'shots_luck', 'experience']
Base features (7): ['pts_pct_z_lag1', 'pts_pct_z_lag2', 'pts_pct_z_lag3', 'pts_pct_z_lag4', 'pts_pct_z_lag5', 'pts_pct_z_mean3', 'pts_pct_z_mean5']
External features (9): ['pts_pct_z_lag1', 'sh_pct_lag1_z', 'sv_pct_lag1_z', 'pdo_lag1_z', 'shot_diff_lag1_z', 'shot_diff_pct_lag1_z', 'pdo_mean3_z', 'shot_diff_pct_mean3_z', 'n_hist']


## 2. Models

Both models output a score per team; the ranking is obtained by sorting the scores inside each
`(season, conference)`. Hyper-parameters are fixed to conservative values; nothing is tuned on the
test seasons.

* **Ridge + external**: standardised features, L2 regularisation with the penalty chosen by
  internal cross-validation on the training seasons only.
* **Random forest + external**: shallow trees (`max_depth=4`, `min_samples_leaf=20`,
  `max_features=0.7`, 500 trees). The forest can capture non-linearities and interactions that
  Ridge cannot, at the cost of more variance; the depth and leaf size keep each tree simple.

In [2]:
# ---- model factories -------------------------------------------------------
ridge = lambda: make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))
random_forest = lambda: RandomForestRegressor(
    n_estimators=500, max_depth=4, min_samples_leaf=20,
    max_features=0.7, n_jobs=-1, random_state=RS,
)

MODELS = {
    "Ridge + external": {
        "family": "External",
        "features": EXT_FEATURES,
        "data": "external",
        "factory": ridge,
    },
    "Random forest + external": {
        "family": "External",
        "features": EXT_FEATURES,
        "data": "external",
        "factory": random_forest,
    },
}

NAIVE = {
    "Naive: random": "Random scores, fixed seed (no skill, floor).",
    "Naive: previous available ranking": (
        "Last available season's points percentage, re-ranked in the current conference; "
        "league average when the team has no history."
    ),
}
FAMILY = {name: "Naive" for name in NAIVE}
FAMILY.update({name: cfg["family"] for name, cfg in MODELS.items()})


print(f"{len(NAIVE)} naive baselines + {len(MODELS)} models")
for name, cfg in MODELS.items():
    print(f"  {cfg['family']:9s} {name:30s} {len(cfg['features'])} features ({cfg['data']})")

2 naive baselines + 2 models
  External  Ridge + external               9 features (external)
  External  Random forest + external       9 features (external)


## 2.1 Blend selection (1995–2004 only)

The blend is a weighted average of the naive previous-ranking score and the best trained model,
after z-scoring both scores by season. Both the model and the weight are chosen on the selection
seasons 1995–2004 (2004 is missing from the data). The pair is then **frozen** and applied unchanged
to the clean test and to the backtest.

The grid is coarse on purpose (steps of 0.05): a fine grid on nine seasons would overfit the
selection split.

In [3]:
SELECTION_YEARS = [y for y in range(1995, 2005) if y != 2004]


def model_predictions(years):
    '''Walk-forward predictions of the naive reference and the two trained models.'''
    preds = {
        "Naive: previous available ranking":
            ru.baseline_scores(df, "pts_pct_z_lag1", years),
    }
    for name, cfg in MODELS.items():
        data = df_ext if cfg["data"] == "external" else df
        preds[name] = ru.walk_forward(
            data, cfg["factory"], cfg["features"], years
        )
    return preds


def zscore_by_year(pred):
    '''Z-score the model score inside each season, indexed by (year, tmID).'''
    p = pred.copy()
    grouped = p.groupby("year")["score"]
    std = grouped.transform("std").replace(0, np.nan)
    p["z"] = ((p["score"] - grouped.transform("mean")) / std).fillna(0.0)
    return p.set_index(["year", "tmID"])["z"]


def rank_mae(pred):
    return ru.season_metrics(ru.rank_predictions(pred, df))["mae"].mean()


selection_preds = model_predictions(SELECTION_YEARS)
selection_mae = {name: rank_mae(p) for name, p in selection_preds.items()}
display(
    pd.Series(selection_mae).sort_values().rename("selection MAE")
    .to_frame().round(3)
)

BLEND_MODEL = min(MODELS, key=selection_mae.get)
naive_ref = selection_preds["Naive: previous available ranking"]
z_naive = zscore_by_year(naive_ref)
z_model = zscore_by_year(selection_preds[BLEND_MODEL])
index = pd.MultiIndex.from_frame(naive_ref[["year", "tmID"]])

grid = {}
for w in np.round(np.arange(0, 1.0001, 0.05), 2):
    blend = naive_ref[["year", "tmID", "confID"]].copy()
    blend["score"] = (w * z_model.reindex(index).to_numpy()
                      + (1 - w) * z_naive.reindex(index).to_numpy())
    grid[float(w)] = rank_mae(blend)

BLEND_WEIGHT = min(grid, key=grid.get)
BLEND_NAME = "Blend: naive + best model"
FAMILY[BLEND_NAME] = "Blend"

print(f"Best model on selection: {BLEND_MODEL} "
      f"(MAE {selection_mae[BLEND_MODEL]:.3f})")
print(f"Best blend weight: w={BLEND_WEIGHT:.2f} "
      f"[blend = w * model + (1-w) * naive], selection MAE {grid[BLEND_WEIGHT]:.3f}")
print("Blend MAE over the weight grid:")
display(pd.Series(grid).round(3).to_frame("MAE"))

,selection MAE
Ridge + external,2.611
Random forest + external,2.633
Naive: previous available ranking,2.879


Best model on selection: Ridge + external (MAE 2.611)
Best blend weight: w=0.95 [blend = w * model + (1-w) * naive], selection MAE 2.604
Blend MAE over the weight grid:


,MAE
0.00,2.879
0.05,2.871
0.10,2.872
0.15,2.840
0.20,2.824
0.25,2.832
0.30,2.833
0.35,2.817
0.40,2.793
0.45,2.755


## 3. Walk-forward predictions

For each split, every model is retrained for every test season. The random baseline is redrawn with
the same seed in both splits. Predicted ranks, errors and the true ranks are stored together, so the
analysis notebook does not need the original data.

In [4]:
def predict_split(split, years):
    preds = {}

    rng = np.random.default_rng(RS)
    random_pred = df.loc[df["year"].isin(years), ["year", "tmID", "confID"]].copy()
    random_pred["score"] = rng.normal(size=len(random_pred))
    preds["Naive: random"] = random_pred

    preds["Naive: previous available ranking"] = ru.baseline_scores(
        df, "pts_pct_z_lag1", years
    )

    preds.update(model_predictions(years))

    # Frozen blend: same model and weight as selected on 1995-2004.
    naive_ref = preds["Naive: previous available ranking"]
    blend = naive_ref[["year", "tmID", "confID"]].copy()
    blend_index = pd.MultiIndex.from_frame(blend[["year", "tmID"]])
    blend["score"] = (
        BLEND_WEIGHT * zscore_by_year(preds[BLEND_MODEL]).reindex(blend_index).to_numpy()
        + (1 - BLEND_WEIGHT) * zscore_by_year(naive_ref).reindex(blend_index).to_numpy()
    )
    preds[BLEND_NAME] = blend

    frames = []
    for name, pred in preds.items():
        ranked = ru.rank_predictions(pred, df)
        ranked["split"] = split
        ranked["model"] = name
        ranked["family"] = FAMILY[name]
        frames.append(ranked[[
            "split", "model", "family", "year", "tmID", "confID",
            "score", "pred_rank", "conf_rank", "conf_size",
            "target", "err", "abs_err",
        ]])
    return pd.concat(frames, ignore_index=True)


predictions = pd.concat(
    [predict_split(split, years) for split, years in SPLITS.items()],
    ignore_index=True,
)

print(f"Prediction rows: {len(predictions)}")
print(predictions.groupby(["split", "model"]).size().unstack(fill_value=0).T)

Prediction rows: 2765
split                              backtest  clean
model                                             
Blend: naive + best model               403    150
Naive: previous available ranking       403    150
Naive: random                           403    150
Random forest + external                403    150
Ridge + external                        403    150


## 4. Metrics and saved results

`metrics_by_season.csv` keeps one row per ranking, which is the right unit for the paired bootstrap
in the analysis notebook. `summary.csv` is the leaderboard.

In [5]:
def pair_accuracy(r):
    taus = [
        kendalltau(g["pred_rank"], g["conf_rank"])[0]
        for _, g in r.groupby(["year", "confID"])
    ]
    return (1 + np.mean(taus)) / 2


season_rows = []
for (split, model), r in predictions.groupby(["split", "model"]):
    table = ru.season_metrics(r)
    table["split"] = split
    table["model"] = model
    table["family"] = FAMILY[model]
    season_rows.append(table)
metrics_by_season = pd.concat(season_rows, ignore_index=True)

summary = (
    metrics_by_season
    .groupby(["split", "model", "family"])
    .agg(
        mae=("mae", "mean"),
        rmse=("rmse", "mean"),
        spearman=("spearman", "mean"),
        top_overlap=("top_overlap", "mean"),
        within_2=("within_2", "mean"),
        n_rankings=("mae", "size"),
    )
    .reset_index()
)
summary["pair_accuracy"] = [
    pair_accuracy(predictions[(predictions["split"] == row.split) &
                              (predictions["model"] == row.model)])
    for row in summary.itertuples()
]
summary = summary.sort_values(["split", "mae", "spearman"], ascending=[True, True, False])

# ---- save ------------------------------------------------------------------
ru.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
predictions.to_csv(ru.RESULTS_DIR / "predictions.csv", index=False, float_format="%.6f")
metrics_by_season.to_csv(ru.RESULTS_DIR / "metrics_by_season.csv", index=False, float_format="%.6f")
summary.to_csv(ru.RESULTS_DIR / "summary.csv", index=False, float_format="%.6f")

run_info = {
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "python": platform.python_version(),
    "splits": {split: list(years) for split, years in SPLITS.items()},
    "selected_groups": SELECTED_GROUPS,
    "base_features": BASE_FEATURES,
    "external_features": EXT_FEATURES,
    "naive": NAIVE,
    "blend": {
        "name": BLEND_NAME,
        "model": BLEND_MODEL,
        "weight": float(BLEND_WEIGHT),
        "selection_years": SELECTION_YEARS,
        "selection_mae": float(grid[BLEND_WEIGHT]),
    },
    "models": {
        name: {"family": cfg["family"], "data": cfg["data"],
               "n_features": len(cfg["features"]), "features": cfg["features"]}
        for name, cfg in MODELS.items()
    },
}
(ru.RESULTS_DIR / "run_info.json").write_text(
    json.dumps(run_info, indent=2), encoding="utf-8"
)

clean = summary[summary["split"] == "clean"].copy()
best = clean.iloc[0]
print(f"Saved results to {ru.RESULTS_DIR}")
print(f"Best clean model: {best['model']} | MAE {best['mae']:.3f} | "
      f"Spearman {best['spearman']:.3f} | top-8 {best['top_overlap']:.0%}")
display(
    clean[["family", "model", "mae", "rmse", "spearman", "top_overlap",
           "within_2", "pair_accuracy"]]
    .set_index(["family", "model"])
    .style.format({
        "mae": "{:.3f}", "rmse": "{:.3f}", "spearman": "{:.3f}",
        "top_overlap": "{:.1%}", "within_2": "{:.1%}", "pair_accuracy": "{:.1%}",
    })
)

Saved results to C:\Users\barcl\Work\Porto\AC\AC_hockey_predictions\ranking\results
Best clean model: Naive: previous available ranking | MAE 3.400 | Spearman 0.477 | top-8 63%


## 5. What to do next

The result files are the input of `ranking_results_analysis.ipynb`. Re-run this notebook only if
the data, the selected features or the model list change; the analysis notebook is cheap and can be
re-run on its own.